# 📊 Post-Training Evaluation & Visualization

This notebook focuses on **Step 5: Analysis**. We will load a trained model checkpoint and perform two critical tasks:
1. **Latent Space Visualization**: Using UMAP to see how nodes have clustered.
2. **Metric Refinement**: Calculating ROC-AUC and Hits@K on the hold-out test set.

In [ ]:
import os
import sys
sys.path.append('.')

import torch
import matplotlib.pyplot as plt
import seaborn as sns
import umap.umap_ as umap

from models.hetero_gnn import BioBridgeLinkPredictor
from data.biobridge_gnn_datamodule import BioBridgeGNNDataModule

sns.set_theme(style="white")

In [ ]:
# 1. Initialize data and model
datamodule = BioBridgeGNNDataModule(data_dir='data/processed/')
datamodule.setup()

model = BioBridgeLinkPredictor(
    metadata=datamodule.data.metadata(),
    hidden_channels=128
)

model.eval()
print("Model and Data loaded for evaluation.")

## 🧬 UMAP Projection
We extract the 128-dimensional embeddings and project them to 2D. In a well-trained model, nodes that interact biologically will form distinct clusters.

In [ ]:
with torch.no_grad():
    z_dict = model.encoder(datamodule.data.x_dict, datamodule.data.edge_index_dict)

def plot_umap(embeddings, title):
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding_2d = reducer.fit_transform(embeddings)
    
    plt.figure(figsize=(10, 7))
    plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c='magenta', alpha=0.5, s=10)
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.show()

print("Visualizing Drug Latent Space...")
plot_umap(z_dict['drug'].numpy()[:500], "BioBridge Drug Embeddings (Latent space)")